In [5]:
import os

import psycopg2
import pandas as pd

# Configurações do Postgres (banco "kronoos")
DB_PARAMS = {
    "host":     os.environ.get("PGHOST",     "10.210.10.15"),
    "port":     os.environ.get("PGPORT",     "5432"),
    "user":     os.environ.get("PGUSER",     "postgres"),
    "password": os.environ.get("PGPASSWORD", "sofkronoos@@2020"),
    "dbname":   os.environ.get("PGDATABASE", "kronoos"),
}

SCRIPT_NAME = "relatorio-certidoes-pedidos"

conn = psycopg2.connect(**DB_PARAMS)
print("Conectado ao Postgres — banco 'kronoos'.")


Conectado ao Postgres — banco 'kronoos'.


In [6]:
# Introspecção de schema: descobre as colunas reais das 4 tabelas e tenta detectar
# automaticamente as foreign keys relevantes, já que o schema não está documentado
# neste repositório.
#
# Cadeia de relacionamento (confirmada pelo usuário):
#   CertidaoPedido -> CertidaoPedidoItem -> CertidaoPedidoItemSolicitacao

TABELAS = ["CertidaoPedido", "CertidaoPedidoItem", "CertidaoPedidoItemSolicitacao", "Usuario"]

SQL_COLUNAS = """
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = %s
    ORDER BY ordinal_position
"""

print("=== Colunas por tabela ===")
colunas_por_tabela = {}
with conn.cursor() as cur:
    for tabela in TABELAS:
        cur.execute(SQL_COLUNAS, (tabela,))
        cols = [c.name for c in cur.description]
        rows = [dict(zip(cols, row)) for row in cur.fetchall()]
        colunas_por_tabela[tabela] = rows
        print(f"\n--- {tabela} ---")
        display(pd.DataFrame(rows))

SQL_FKS = """
    SELECT
        tc.table_name   AS tabela_origem,
        kcu.column_name AS coluna_origem,
        ccu.table_name  AS tabela_destino,
        ccu.column_name AS coluna_destino
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name AND tc.table_schema = kcu.table_schema
    JOIN information_schema.constraint_column_usage ccu
        ON tc.constraint_name = ccu.constraint_name AND tc.table_schema = ccu.table_schema
    WHERE tc.constraint_type = 'FOREIGN KEY' AND tc.table_schema = 'public'
      AND tc.table_name = ANY(%s)
"""

print("\n=== Foreign keys detectadas (origem -> destino) ===")
with conn.cursor() as cur:
    cur.execute(SQL_FKS, (TABELAS,))
    cols = [c.name for c in cur.description]
    fks = [dict(zip(cols, row)) for row in cur.fetchall()]
display(pd.DataFrame(fks))


def _achar_fk(tabela_origem, tabela_destino, campo, default=None):
    return next(
        (f[campo] for f in fks if f["tabela_origem"] == tabela_origem and f["tabela_destino"] == tabela_destino),
        default,
    )


fk_item_pedido = _achar_fk("CertidaoPedidoItem", "CertidaoPedido", "coluna_origem")
print(f"\nFK sugerida (CertidaoPedidoItem -> CertidaoPedido): {fk_item_pedido!r}")

fk_solicitacao_item = _achar_fk("CertidaoPedidoItemSolicitacao", "CertidaoPedidoItem", "coluna_origem")
print(f"FK sugerida (CertidaoPedidoItemSolicitacao -> CertidaoPedidoItem): {fk_solicitacao_item!r}")

fk_usuario = _achar_fk("CertidaoPedido", "Usuario", "coluna_destino", default="id")
print(f"Coluna alvo em Usuario para o join com CertidaoPedido.usuario_aceite: {fk_usuario!r}")

# Heurística para sugerir a coluna de "tipo de certidão" — pode estar em CertidaoPedidoItem
# (o item em si) ou em CertidaoPedidoItemSolicitacao (a solicitação); não dá para detectar
# isso via FK, é um campo de dado, então procuramos nas duas tabelas.
PADROES_TIPO = ("tipo", "certidao", "nome", "descricao", "especie")
candidatas_tipo_item = [
    c["column_name"] for c in colunas_por_tabela["CertidaoPedidoItem"]
    if any(p in c["column_name"].lower() for p in PADROES_TIPO)
]
candidatas_tipo_solicitacao = [
    c["column_name"] for c in colunas_por_tabela["CertidaoPedidoItemSolicitacao"]
    if any(p in c["column_name"].lower() for p in PADROES_TIPO)
]
print(f"\nColunas candidatas para 'tipo de certidão' em CertidaoPedidoItem: {candidatas_tipo_item}")
print(f"Colunas candidatas para 'tipo de certidão' em CertidaoPedidoItemSolicitacao: {candidatas_tipo_solicitacao}")

print("\n=== Amostra de CertidaoPedidoItem (5 linhas, para conferência visual) ===")
with conn.cursor() as cur:
    cur.execute('SELECT * FROM "CertidaoPedidoItem" LIMIT 5')
    cols = [c.name for c in cur.description]
    display(pd.DataFrame([dict(zip(cols, row)) for row in cur.fetchall()]))

print("\n=== Amostra de CertidaoPedidoItemSolicitacao (5 linhas, para conferência visual) ===")
with conn.cursor() as cur:
    cur.execute('SELECT * FROM "CertidaoPedidoItemSolicitacao" LIMIT 5')
    cols = [c.name for c in cur.description]
    display(pd.DataFrame([dict(zip(cols, row)) for row in cur.fetchall()]))


=== Colunas por tabela ===

--- CertidaoPedido ---


,column_name,data_type,is_nullable
0,id,integer,NO
1,titulo,character varying,NO
2,id_solicitante,integer,NO
3,id_usuario_solicitante,integer,NO
4,id_usuario_aceite,integer,YES
5,status,USER-DEFINED,NO
6,created_at,timestamp with time zone,NO
7,updated_at,timestamp with time zone,NO
8,id_pedido,integer,YES
9,organizacao_id,integer,NO



--- CertidaoPedidoItem ---


,column_name,data_type,is_nullable
0,id,integer,NO
1,id_pedido,integer,NO
2,dados,json,NO
3,created_at,timestamp with time zone,NO
4,updated_at,timestamp with time zone,NO



--- CertidaoPedidoItemSolicitacao ---


...

,tabela_origem,coluna_origem,tabela_destino,coluna_destino
0,Usuario,id_departamento,Departamento,id
1,Usuario,idioma_preferido,Idiomas,codigo
2,Usuario,organizacao_id,Organizacao,id
3,CertidaoPedido,id_dossie,Pedido,id
4,CertidaoPedido,id_pedido,Pedido,id
5,CertidaoPedido,id_solicitante,CertidaoSolicitante,id
6,CertidaoPedido,id_usuario_aceite,Usuario,id
7,CertidaoPedido,id_usuario_solicitante,Usuario,id
8,CertidaoPedidoItemSolicitacao,id_certidao,Certidao,id
9,CertidaoPedidoItemSolicitacao,id_item,CertidaoPedidoItem,id



FK sugerida (CertidaoPedidoItem -> CertidaoPedido): 'id_pedido'
FK sugerida (CertidaoPedidoItemSolicitacao -> CertidaoPedidoItem): 'id_item'
Coluna alvo em Usuario para o join com CertidaoPedido.usuario_aceite: 'id'

Colunas candidatas para 'tipo de certidão' em CertidaoPedidoItem: []
Colunas candidatas para 'tipo de certidão' em CertidaoPedidoItemSolicitacao: ['id_certidao']

=== Amostra de CertidaoPedidoItem (5 linhas, para conferência visual) ===


,id,id_pedido,dados,created_at,updated_at
0,1,1,"{'matricula': 'JND30081033', 'cartorio': 'Cart...",2024-12-24 11:44:21.648000-03:00,2024-12-24 11:44:21.648000-03:00
1,2,2,"{'cpf': '435.950.098-00', 'rg': '45.285.585-8'...",2025-01-10 15:35:03.009000-03:00,2025-01-10 15:35:03.009000-03:00
2,3,3,"{'cpf': '435.950.098-00', 'rg': '45.285.585-8'...",2025-01-15 12:02:03.462000-03:00,2025-01-15 12:02:03.462000-03:00
3,4,4,"{'cpf': '222.222.222-22', 'rg': '34285541', 'n...",2025-01-20 09:25:46.056000-03:00,2025-01-20 09:25:46.056000-03:00
4,5,5,"{'matricula': '123456', 'cartorio': 'teste', '...",2025-01-22 14:00:18.582000-03:00,2025-01-22 14:00:18.582000-03:00



=== Amostra de CertidaoPedidoItemSolicitacao (5 linhas, para conferência visual) ===


,id,id_item,id_certidao,data_emissao,data_validade,resultado,status,valor,created_at,updated_at,observacoes,protocolo,cobrar,pendencia,cliente_observacoes,dados_adicionais
0,13,3,371,2025-01-15 00:00:00-03:00,2025-04-15 00:00:00-03:00,SIM,FINALIZADO,5.9,2025-01-15 12:02:03.508000-03:00,2025-01-15 12:25:32.108000-03:00,NaN,None,True,None,None,None
1,14,3,692,NaT,NaT,NAO,FINALIZADO,5.9,2025-01-15 12:02:03.508000-03:00,2025-01-15 12:26:17.919000-03:00,CPF não encontrado,None,True,None,None,None
2,15,3,55,2025-01-15 00:00:00-03:00,2025-04-15 00:00:00-03:00,SIM,FINALIZADO,5.9,2025-01-15 12:02:03.509000-03:00,2025-01-15 12:28:30.654000-03:00,NaN,None,True,None,None,None
3,16,3,366,2025-01-15 00:00:00-03:00,2025-04-15 00:00:00-03:00,SIM,FINALIZADO,5.9,2025-01-15 12:02:03.509000-03:00,2025-01-15 12:29:27.080000-03:00,NaN,None,True,None,None,None
4,17,3,354,2025-01-15 00:00:00-03:00,2025-04-15 00:00:00-03:00,SIM,FINALIZADO,5.9,2025-01-15 12:02:03.509000-03:00,2025-01-15 12:31:14.913000-03:00,NaN,None,True,None,None,None


In [7]:
# CONFIG — ajuste manualmente se a introspecção da Cell 2 não detectar com confiança
# (confira os prints e as tabelas exibidas acima antes de rodar o restante do notebook).

COL_ITEM_FK_PEDIDO = fk_item_pedido            # coluna em CertidaoPedidoItem -> CertidaoPedido.id
COL_SOLICITACAO_FK_ITEM = fk_solicitacao_item  # coluna em CertidaoPedidoItemSolicitacao -> CertidaoPedidoItem.id
COL_USUARIO_ID = fk_usuario                    # coluna em Usuario usada no join com CertidaoPedido.usuario_aceite

# Coluna (e tabela) que identifica o tipo/nome da certidão, usada no ranking de
# "certidões mais pedidas". Pode estar em CertidaoPedidoItem ou em
# CertidaoPedidoItemSolicitacao — veja as candidatas e as amostras impressas na Cell 2
# e ajuste aqui se a sugestão automática não fizer sentido.
if candidatas_tipo_item:
    TABELA_TIPO_CERTIDAO, COL_TIPO_CERTIDAO = "CertidaoPedidoItem", candidatas_tipo_item[0]
elif candidatas_tipo_solicitacao:
    TABELA_TIPO_CERTIDAO, COL_TIPO_CERTIDAO = "CertidaoPedidoItemSolicitacao", candidatas_tipo_solicitacao[0]
else:
    TABELA_TIPO_CERTIDAO, COL_TIPO_CERTIDAO = None, None

# Se cada linha de CertidaoPedidoItemSolicitacao já representa 1 certidão, deixe None
# (a quantidade por pedido = COUNT(*) de linhas). Se existir uma coluna de quantidade
# própria, informe o nome aqui (a quantidade passa a ser SUM(coluna)).
COL_QTD_CERTIDAO = None

assert COL_ITEM_FK_PEDIDO, (
    "Não foi possível detectar a FK de CertidaoPedidoItem -> CertidaoPedido. "
    "Preencha COL_ITEM_FK_PEDIDO manualmente com o nome da coluna correta."
)
assert COL_SOLICITACAO_FK_ITEM, (
    "Não foi possível detectar a FK de CertidaoPedidoItemSolicitacao -> CertidaoPedidoItem. "
    "Preencha COL_SOLICITACAO_FK_ITEM manualmente com o nome da coluna correta."
)
assert COL_TIPO_CERTIDAO, (
    "Não foi possível sugerir a coluna de tipo de certidão. "
    "Preencha TABELA_TIPO_CERTIDAO e COL_TIPO_CERTIDAO manualmente."
)

print(f"COL_ITEM_FK_PEDIDO      = {COL_ITEM_FK_PEDIDO!r}")
print(f"COL_SOLICITACAO_FK_ITEM = {COL_SOLICITACAO_FK_ITEM!r}")
print(f"COL_USUARIO_ID          = {COL_USUARIO_ID!r}")
print(f"TABELA_TIPO_CERTIDAO    = {TABELA_TIPO_CERTIDAO!r}")
print(f"COL_TIPO_CERTIDAO       = {COL_TIPO_CERTIDAO!r}")
print(f"COL_QTD_CERTIDAO        = {COL_QTD_CERTIDAO!r}")


COL_ITEM_FK_PEDIDO      = 'id_pedido'
COL_SOLICITACAO_FK_ITEM = 'id_item'
COL_USUARIO_ID          = 'id'
TABELA_TIPO_CERTIDAO    = 'CertidaoPedidoItemSolicitacao'
COL_TIPO_CERTIDAO       = 'id_certidao'
COL_QTD_CERTIDAO        = None


In [8]:
# Query de detalhe: uma linha por pedido (CertidaoPedido), contando certidões através
# de CertidaoPedidoItem -> CertidaoPedidoItemSolicitacao.

def is_reprocessamento(valor):
    """Trata reprocessamento como booleano mesmo que o banco guarde bool/int/string
    (ex.: True, 1, 't', 'sim') — o tipo real da coluna não pôde ser confirmado."""
    if valor is None:
        return False
    if isinstance(valor, bool):
        return valor
    return str(valor).strip().lower() in ("1", "true", "t", "s", "sim", "yes")


qtd_expr = (
    f'COUNT(cpis."{COL_SOLICITACAO_FK_ITEM}")' if COL_QTD_CERTIDAO is None
    else f'SUM(cpis."{COL_QTD_CERTIDAO}")'
)

SQL_DETALHE = f"""
    SELECT
        cp.id AS id_tarefa,
        cp.status AS status,
        cp.reprocessamento AS reprocessamento,
        u.nome AS usuario_nome,
        u.email AS usuario_email,
        {qtd_expr} AS qtd_certidoes
    FROM "CertidaoPedido" cp
    LEFT JOIN "Usuario" u ON u."{COL_USUARIO_ID}" = cp.id_usuario_aceite
    LEFT JOIN "CertidaoPedidoItem" cpi ON cpi."{COL_ITEM_FK_PEDIDO}" = cp.id
    LEFT JOIN "CertidaoPedidoItemSolicitacao" cpis ON cpis."{COL_SOLICITACAO_FK_ITEM}" = cpi.id
    GROUP BY cp.id, cp.status, cp.reprocessamento, u.nome, u.email
    ORDER BY cp.id
"""

with conn.cursor() as cur:
    cur.execute(SQL_DETALHE)
    cols = [c.name for c in cur.description]
    rows = [dict(zip(cols, row)) for row in cur.fetchall()]

df_detalhe = pd.DataFrame(rows, columns=["id_tarefa", "status", "reprocessamento", "usuario_nome", "usuario_email", "qtd_certidoes"])
df_detalhe["Tipo de Tarefa"] = df_detalhe["reprocessamento"].apply(
    lambda v: "Reprocessamento" if is_reprocessamento(v) else "Normal"
)
df_detalhe["Usuário"] = df_detalhe["usuario_nome"].fillna("Não aceito")
df_detalhe["Email"] = df_detalhe["usuario_email"].fillna("")
df_detalhe["Qtd Certidões"] = df_detalhe["qtd_certidoes"].fillna(0).astype(int)

df_detalhe = df_detalhe.rename(columns={"id_tarefa": "Id da Tarefa", "status": "Status"})
df_detalhe = df_detalhe[["Id da Tarefa", "Qtd Certidões", "Usuário", "Email", "Status", "Tipo de Tarefa"]]

print(f"{len(df_detalhe)} pedido(s) carregado(s).")
display(df_detalhe.head())


71574 pedido(s) carregado(s).


,Id da Tarefa,Qtd Certidões,Usuário,Email,Status,Tipo de Tarefa
0,1,1,Bruno Xavier de Melo | Kronoos,bruno.melo@kronoos.com,FINALIZADO,Normal
1,2,3,Guilherme Miranda | Kronoos,guilherme.miranda@kronoos.com,FINALIZADO,Normal
2,3,64,Guilherme Miranda | Kronoos,guilherme.miranda@kronoos.com,FINALIZADO,Normal
3,4,10,Alexandre Pegoraro,alexandre.pegoraro@kronoos.com,FINALIZADO,Normal
4,5,3,Guilherme Miranda | Kronoos,guilherme.miranda@kronoos.com,FINALIZADO,Normal


In [9]:
# Query de certidões mais pedidas (ranking por tipo/nome de certidão).
# A coluna pode estar em CertidaoPedidoItem ou em CertidaoPedidoItemSolicitacao
# (ver CONFIG na Cell 3).

SQL_MAIS_PEDIDAS = f"""
    SELECT "{COL_TIPO_CERTIDAO}" AS tipo_certidao, COUNT(*) AS qtd
    FROM "{TABELA_TIPO_CERTIDAO}"
    GROUP BY "{COL_TIPO_CERTIDAO}"
    ORDER BY qtd DESC
"""

with conn.cursor() as cur:
    cur.execute(SQL_MAIS_PEDIDAS)
    cols = [c.name for c in cur.description]
    rows = [dict(zip(cols, row)) for row in cur.fetchall()]

df_mais_pedidas = pd.DataFrame(rows, columns=["tipo_certidao", "qtd"]).rename(
    columns={"tipo_certidao": "Tipo de Certidão", "qtd": "Qtd Solicitações"}
)
display(df_mais_pedidas)


,Tipo de Certidão,Qtd Solicitações
0,2428,10475
1,1718,10296
2,2423,10102
3,2427,10094
4,1353,5479
...,...,...
1165,1040,1
1166,1628,1
1167,1148,1
1168,2639,1


In [10]:
# Agregações do Resumo (a partir do df_detalhe já carregado).

total_pedidos = len(df_detalhe)
total_certidoes = int(df_detalhe["Qtd Certidões"].sum())
qtd_reprocessamento = int((df_detalhe["Tipo de Tarefa"] == "Reprocessamento").sum())
qtd_normal = total_pedidos - qtd_reprocessamento

df_resumo_geral = pd.DataFrame([
    {"Métrica": "Total de Pedidos", "Valor": total_pedidos},
    {"Métrica": "Total de Certidões", "Valor": total_certidoes},
    {"Métrica": "Pedidos com Reprocessamento", "Valor": qtd_reprocessamento},
    {"Métrica": "Pedidos sem Reprocessamento (Normal)", "Valor": qtd_normal},
])

df_por_status = (
    df_detalhe.groupby("Status").size().reset_index(name="Qtd Pedidos").sort_values("Qtd Pedidos", ascending=False)
)

df_por_usuario = (
    df_detalhe.groupby(["Usuário", "Email"])
    .agg(**{"Qtd Pedidos": ("Id da Tarefa", "count"), "Qtd Certidões": ("Qtd Certidões", "sum")})
    .reset_index()
    .sort_values("Qtd Certidões", ascending=False)
)

print("Resumo geral:")
display(df_resumo_geral)
print("\nPedidos por status:")
display(df_por_status)
print("\nCertidões por usuário:")
display(df_por_usuario)


Resumo geral:


,Métrica,Valor
0,Total de Pedidos,71574
1,Total de Certidões,111767
2,Pedidos com Reprocessamento,66856
3,Pedidos sem Reprocessamento (Normal),4718



Pedidos por status:


,Status,Qtd Pedidos
2,FINALIZADO,71346
3,PENDENCIA,136
0,ACEITO,56
1,EM ANDAMENTO,29
4,SOLICITADO,7



Certidões por usuário:


,Usuário,Email,Qtd Pedidos,Qtd Certidões
9,Jhonanthan Rondina,jhonanthan.rondina@kronoos.com,31511,43108
3,Celso Moreira,celso.moreira@kronoos.com,16820,32358
4,Dalton Alves | Kronoos,dalton.alves@kronoos.com,13172,17736
14,Raul Almeida,raul.almeida@kronoos.com,4373,8573
6,Guilherme Miranda | Kronoos,guilherme.miranda@kronoos.com,4227,5324
8,Janaine Moreira,janaine.moreira@kronoos.com,1020,4049
7,Ianca Dantas | Kronoos,ianca.dantas@kronoos.com,348,484
13,Não aceito,,57,77
1,Bruno Dias,bnxdias@gmail.com,23,25
0,Alexandre Pegoraro,alexandre.pegoraro@kronoos.com,4,13


In [11]:
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

RESPONSES_DIR = os.path.join("..", "responses", SCRIPT_NAME)
os.makedirs(RESPONSES_DIR, exist_ok=True)

THIN_BORDER = Border(
    left=Side(style="thin", color="BFBFBF"), right=Side(style="thin", color="BFBFBF"),
    top=Side(style="thin", color="BFBFBF"), bottom=Side(style="thin", color="BFBFBF"),
)
HEADER_BORDER = Border(
    left=Side(style="thin", color="BFBFBF"), right=Side(style="thin", color="BFBFBF"),
    top=Side(style="thin", color="BFBFBF"), bottom=Side(style="medium", color="808080"),
)


def format_sheet(ws, df, header_hex="D9D9D9", startrow=0, freeze_and_filter=False):
    """Cabeçalho colorido, bordas finas e largura automática de coluna para uma tabela
    que começa em `startrow` (0-indexado, mesmo parâmetro do df.to_excel). Freeze panes
    e autofiltro são opcionais — só fazem sentido para uma tabela única por planilha."""
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    data_align = Alignment(horizontal="left", vertical="center", wrap_text=False)
    header_row = startrow + 1

    for col_idx, col_name in enumerate(df.columns, start=1):
        header_cell = ws.cell(row=header_row, column=col_idx)
        header_cell.fill = PatternFill("solid", fgColor=header_hex)
        header_cell.font = Font(bold=True, color="3B3B3B", size=10)
        header_cell.alignment = header_align
        header_cell.border = HEADER_BORDER

        valores = df[col_name] if len(df) else []
        max_len = max([len(str(col_name))] + [len(str(v)) for v in valores]) if len(df) else len(str(col_name))
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 4, 12), 60)

        for row_idx in range(header_row + 1, header_row + len(df) + 1):
            data_cell = ws.cell(row=row_idx, column=col_idx)
            data_cell.alignment = data_align
            data_cell.border = THIN_BORDER

    if freeze_and_filter:
        ws.row_dimensions[header_row].height = 22
        ws.freeze_panes = ws.cell(row=header_row + 1, column=1).coordinate
        ws.auto_filter.ref = f"A{header_row}:{get_column_letter(len(df.columns))}{header_row + len(df)}"


print(f"Função de formatação carregada. Diretório de saída: {RESPONSES_DIR}")


Função de formatação carregada. Diretório de saída: ../responses/relatorio-certidoes-pedidos


In [12]:
# Escrita do Excel final: aba "Resumo" (blocos empilhados) + aba "Detalhes".

output_file = os.path.join(RESPONSES_DIR, "relatorio_certidoes_pedidos.xlsx")

BLOCOS_RESUMO = [
    ("Resumo Geral", df_resumo_geral, "FFD966"),
    ("Pedidos por Status", df_por_status, "9DC3E6"),
    ("Certidões por Usuário", df_por_usuario, "A9D18E"),
    ("Certidões Mais Pedidas", df_mais_pedidas, "F4B183"),
]

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    row_cursor = 0
    ws_resumo = None
    for titulo, df_bloco, cor in BLOCOS_RESUMO:
        titulo_row = row_cursor + 1       # linha (1-indexada) do título do bloco
        table_startrow = row_cursor + 1   # startrow (0-indexado) do df.to_excel -> cabeçalho na linha seguinte ao título
        df_bloco.to_excel(writer, sheet_name="Resumo", startrow=table_startrow, index=False)
        if ws_resumo is None:
            ws_resumo = writer.sheets["Resumo"]
        titulo_cell = ws_resumo.cell(row=titulo_row, column=1, value=titulo)
        titulo_cell.font = Font(bold=True, size=12, color="1F4E78")
        format_sheet(ws_resumo, df_bloco, header_hex=cor, startrow=table_startrow)
        row_cursor += len(df_bloco) + 3  # título + cabeçalho + linhas de dados + 1 linha em branco

    df_detalhe.to_excel(writer, sheet_name="Detalhes", index=False)
    format_sheet(writer.sheets["Detalhes"], df_detalhe, header_hex="9DC3E6", freeze_and_filter=True)

conn.close()
print(f"Relatório salvo em: {output_file}")


Relatório salvo em: ../responses/relatorio-certidoes-pedidos/relatorio_certidoes_pedidos.xlsx
